# Запечатанные семейства: обучатель против сетей

`research_m5/cells.py` — один обучатель для любого семейства задач «прочитай последовательность — дай ответ».
Он ищет **маленькую машину**: конечное управление (номер состояния или несколько переключателей) и рядом
до двух чисел (счётчик, уровень с краями, круг). Ходы чисел могут зависеть от того, что числа читают
(проверка счётчика на ноль, как у машины Минского). Числа могут стоять каскадом, как у Крона и Роудса:
первое ведёт второе, и ответ читается со второго.
Машину находит решатель SAT, перебирая формы в порядке длины описания в битах. Потом обучатель **сам
спрашивает**: решатель ищет вторую машину, которая тоже объясняет всё выученное, вместе с рассказом, на
котором они расходятся, и обучатель спрашивает ответ ровно на этот рассказ. Если вопросы кончились раньше,
чем машина прошла последнюю проверку, или в окне осталась вторая машина, ответ выдаётся с меткой
**«не уверен»**.

**Протокол** (README, этапы 53, 58, 60, 64, 67). Каждое семейство написал отдельный исполнитель, который видел только
короткое задание (`sealed/SPEC_SYMBOLS.md`) и тему и ничего не знал об обучателе. Семейства и их хеши
записаны в репозиторий до прогона; прогон — один, замороженным кодом. Учение — 300 последовательностей
длиной 1–4 и не больше 40 вопросов; проверка — 300 последовательностей длиной 20, 100 и 500.

**Что было записано** (95 % и выше на длине 100):

| пул | обучатель | из них точно 100 / 100 / 100 | молчаливые ошибки | лучшая сеть (LSTM / трансформер, до 12 000 рассказов) |
|---|---|---|---|---|
| 53 | 2 из 12 | 1 | 1 | — |
| 58 | 6 из 12 | 5 | 1 | 1 из 12 |
| 60 | 7 из 12 | 7 | 0 | 3 из 12 |
| 64 | 6 из 12 | 6 | 1 | 0 из 12 (и ALiBi) |
| 66 | 7 из 12 | 6 | 0 | 2 из 12 |

На пулах 58 и 60 проверены и сети сильнее: ALiBi и трансформер 6×256 на 12 000 рассказов —
ни одного семейства с 95 % на длине 100 (этап 63а).

Обучатель работает на процессоре; решатель ограничен числом конфликтов, а не секундами, поэтому итог не
зависит от машины (кроме предела в 30 минут на семейство). Видеокарта нужна только сетям-соперникам.
Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки.

In [ ]:
import os, subprocess
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
CODE = 'edbc9e9bace97747495c568aee402295629c69c0'  # the code of the recorded run (stage 67), frozen before it
subprocess.run(['git', '-C', ROOT + '/MX-Project', 'checkout', '-q', CODE, '--', 'research_m5/cells.py', 'research_m5/sealed_run.py'], check=True)
!pip -q install python-sat
WORKERS = os.cpu_count()
print('ядер:', WORKERS)

## Обучатель на пуле 66

Код и флаги — те же, что в записанном прогоне этапа 67 (коммит edbc9e9). Семейства идут параллельно, по
одному на ядро. На двух ядрах Colab весь пул занимает около двух часов: большинство семейств кончается за
минуты, два упираются в предел 30 минут. В быстром режиме — четыре семейства. Рядом печатается то, что было записано при
прогоне до этой проверки: при тех же флагах числа должны совпасть (кроме времени).

In [ ]:
import glob, os, subprocess, time
POOL = 'sealed/stage66_families'
FLAGS = '--minutes 30 --workers 1 --seconds 3 --shorten --reserve 21 --explore --frugal --slack 16 --guards --build 12 --flags --cascades'
names = sorted(os.listdir(f'{M5}/{POOL}'))
if QUICK:
    names = ['04-hotel-reception', '11-concert-hall', '01-brewery', '02-sawmill']
groups = [names[i::WORKERS] for i in range(WORKERS)]
os.makedirs('/content/run', exist_ok=True)
procs = []
for g, part in enumerate(groups):
    d = f'/content/run/g{g}'
    os.makedirs(d, exist_ok=True)
    for n in part:
        if not os.path.exists(f'{d}/{n}'):
            os.symlink(f'{M5}/{POOL}/{n}', f'{d}/{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {FLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/run/out{g}.txt', shell=True))
t0 = time.time()
for p in procs:
    p.wait()
print(f'{time.time() - t0:.0f} s')
mine = {}
for f in glob.glob('/content/run/out*.txt'):
    for line in open(f):
        mine[line.split(' ')[0]] = line.strip()
recorded = {line.split(' ')[0]: line.strip() for line in open(f'{M5}/results/stage67/results.txt')}
import re
num = lambda s: re.sub(r'\(\d+ s\)$', '', s)
for n in names:
    print(mine.get(n, n + ': (нет)'))
    print('   записано:', 'то же' if num(mine.get(n, '')) == num(recorded.get(n, '')) else recorded.get(n))

## Соперники на видеокарте

LSTM, трансформер и ALiBi на тех же семействах: учение на тех же 300 коротких рассказах и на в 40 раз
большем числе (12 000), три зерна, 3000 шагов; плюс трансформер и ALiBi побольше (6 слоёв, ширина 256).
Засчитывается лучшее. Записанное на процессоре (этап 67): 2 из 12 (пивоварня и шлюз канала, оба на 12 000 рассказах; оба взяты и обучателем).

In [ ]:
fams = ' '.join(names)
!cd {M5} && python cells_rivals.py --folder sealed/stage66_families {fams} --device cuda --kinds lstm transformer alibi 2>/dev/null | grep "mean of"
!cd {M5} && python cells_rivals.py --folder sealed/stage66_families {fams} --device cuda --kinds transformer alibi --width 256 --layers 6 --times 40 2>/dev/null | grep "mean of"

## Своё семейство

Напишите правило сами — обучатель о нём ничего не знает. Нужны `ALPHABET` (2–8 слов) и `answer(seq)`,
которая для любого списка слов возвращает короткую строку. Ниже пример: лифт на пять этажей, который
нельзя сдвинуть, пока открыты двери.

In [ ]:
os.makedirs('/content/mine/01-my-family', exist_ok=True)
open('/content/mine/01-my-family/family.py', 'w').write('''
ALPHABET = ["up", "down", "open", "close"]
def answer(seq):
    floor, doors = 0, False
    for t in seq:
        if t == "up" and not doors: floor = min(4, floor + 1)
        elif t == "down" and not doors: floor = max(0, floor - 1)
        elif t == "open": doors = True
        elif t == "close": doors = False
    return f"{floor} {'open' if doors else 'shut'}"
''')
!cd {M5} && python sealed_run.py /content/mine {FLAGS} 2>/dev/null